# Next Category Prediction - 60 Days
## Fixed-model training workflow

### Standard model workflow

This notebook uses a deterministic scikit-learn model instead of the experimental Fabric AutoML/FLAML workflow.

In [ ]:
import logging
import warnings
 
logging.getLogger('synapse.ml').setLevel(logging.CRITICAL)
logging.getLogger('mlflow.utils').setLevel(logging.CRITICAL)
warnings.simplefilter('ignore', category=FutureWarning)
warnings.simplefilter('ignore', category=UserWarning)

## Step 1: Load the Data

This cell is responsible for importing the raw data from the specified source into the notebook environment. The data could come from various sources, such as a file or table in your lakehouse.

Once loaded, this data will serve as the input for subsequent steps, such as data transformation, model training, and evaluation.

In [ ]:
import pandas as pd
import numpy as np

feature_cols = [
    "recency", "buy_days", "orders_90", "units_90", "net_90",
    "last_bikes", "last_accessories", "last_other",
]
target_col = "target"

wide = spark.read.format("delta").load("Tables/dbo/ml_wide_next_category_60d")
wide = wide.withColumn("net_90", wide["net_90"].cast("double"))

X = (
    wide.where("data_split = 'train_pool' AND target IS NOT NULL")
    .select(*feature_cols, target_col)
    .toPandas()
)

if X.empty:
    raise ValueError("train_pool không có dòng có nhãn.")

In [ ]:
display(X)

## Step 2: Generate features

Featurization is the process of transforming raw data into a format optimized for training a machine learning model. It ensures the model can access the most relevant information, significantly impacting its accuracy and performance.

This step applies various techniques to refine the data, enhance its quality, and make it compatible with the selected algorithms, helping the model learn patterns more effectively.

In [ ]:
# Handle class imbalance
import matplotlib.pyplot as plt


distribution = X[target_col].value_counts(normalize=True)
dominant_class_proportion = distribution.max()

distribution.plot(kind='bar')
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Proportion")
plt.show()

if dominant_class_proportion > 0.8:
    print(f"The dataset is imbalanced. The dominant class has {dominant_class_proportion * 100:.2f}% of the samples.")
    print("You may need to handle class imbalance before training the model.")
    print("You can use techniques such as oversampling, undersampling, or using class weights to handle class imbalance.")
    print("For more information, see https://aka.ms/smote-example")
else:
    print("The dataset is balanced.")


In [ ]:
# Set Functions if needed for Featurization
def _coerce_na_for_sklearn(df):
    """Replace pd.NA with np.nan in object-like columns (category/string/boolean/object).

    sklearn's SimpleImputer extracts these columns as object arrays and calls
    ``X != X`` to detect NaN, which raises ``TypeError: boolean value of NA is
    ambiguous`` when pandas extension dtypes propagate ``pd.NA``. Converting
    such columns to plain object with ``np.nan`` keeps numeric/datetime
    extension columns untouched.
    """
    df = df.copy()
    for col in df.columns:
        s = df[col]
        dt = s.dtype
        is_object_like = (
            isinstance(dt, pd.CategoricalDtype)
            or dt == object
            or dt == 'boolean'
            or str(dt) == 'string'
        )
        if is_object_like:
            s_obj = s.astype(object)
            df[col] = s_obj.where(s_obj.notna(), np.nan)
    return df


def create_fillna_processor(
    df, mean_features=None, median_features=None, mode_features=None
):
    """
    Create a ColumnTransformer that fills missing values in a DataFrame using different strategies
    based on the skewness of the numerical features and the specified feature lists.

    Parameters:
    df (pd.DataFrame): The input DataFrame.
    mean_features (list, optional): List of features to impute using the mean strategy. Defaults to None.
    median_features (list, optional): List of features to impute using the median strategy. Defaults to None.
    mode_features (list, optional): List of features to impute using the mode strategy. Defaults to None.

    Returns:
    ColumnTransformer: A fitted ColumnTransformer that can be used to transform the DataFrame.
    list: List of all features supported by SimpleImputer in the DataFrame.
    list: List of datetime features in the DataFrame.
    """
    if mean_features is None:
        mean_features = []
    if median_features is None:
        median_features = []
    if mode_features is None:
        mode_features = []
    all_features = mean_features + median_features + mode_features
    # Group features by their imputation needs
    mean_features = [
        col
        for col in df.select_dtypes(include=["number"], exclude=["timedelta"]).columns
        if df[col].skew(skipna=True) <= 1 and col not in all_features
    ] + mean_features
    median_features = [
        col
        for col in df.select_dtypes(include=["number"], exclude=["timedelta"]).columns
        if df[col].skew(skipna=True) > 1 and col not in all_features
    ] + median_features
    all_features = mean_features + median_features
    datetime_features = df.select_dtypes(include=["datetime", "timedelta"]).columns.tolist()
    mode_features = [col for col in df.columns.tolist() if col not in all_features + datetime_features]

    transformers = []

    if mean_features:
        transformers.append(
            ("mean_imputer", SimpleImputer(strategy="mean"), mean_features)
        )
    if median_features:
        transformers.append(
            ("median_imputer", SimpleImputer(strategy="median"), median_features)
        )
    if mode_features:
        transformers.append(
            ("mode_imputer", SimpleImputer(strategy="most_frequent"), mode_features)
        )

    column_transformer = ColumnTransformer(transformers=transformers)
    all_features = mean_features + median_features + mode_features

    return column_transformer.fit(_coerce_na_for_sklearn(df)), all_features, datetime_features


def fillna(df, processor, all_features, datetime_features):
    """
    Fill missing values in a DataFrame using a specified processor and mode imputation.

    Parameters:
    df (pd.DataFrame): The input DataFrame with missing values.
    processor (object): An object with a `transform` method that processes the DataFrame.
    all_features (list): List of all features supported by SimpleImputer in the DataFrame.
    datetime_features (list): List of datetime features in the DataFrame.

    Returns:
    pd.DataFrame: A DataFrame with missing values filled.
    """
    filled_array = processor.transform(_coerce_na_for_sklearn(df))
    filled_df = pd.DataFrame(filled_array, columns=all_features, index=df.index)
    if datetime_features:
        datetime_data = df[datetime_features]
        datetime_data = datetime_data.ffill()
        filled_df = pd.concat([datetime_data, filled_df], axis=1)
    filled_df = filled_df.fillna(filled_df.mode().iloc[0])

    return filled_df


In [ ]:
# Fixed-model workflow; no FLAML dependency is required.


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

train_data = X.loc[:, [*feature_cols, target_col]].copy()
holdout_data = (
    wide.where("data_split = 'holdout' AND target IS NOT NULL")
    .select(*feature_cols, target_col)
    .toPandas()
)

if holdout_data.empty:
    raise ValueError("holdout không có dòng có nhãn.")

y_train = train_data.pop(target_col).astype(str)
y_test = holdout_data.pop(target_col).astype(str)

preprocessor, all_features, datetime_features = create_fillna_processor(train_data)
X_train = fillna(train_data, preprocessor, all_features, datetime_features)
X_test = fillna(holdout_data, preprocessor, all_features, datetime_features)

print(f"Train pool: {len(X_train):,} dòng")
print(f"Holdout: {len(X_test):,} dòng")
print(y_train.value_counts())
display(X_train.head(10))

## Step 3: Train and evaluate a standard multi-class classifier

AutoML/FLAML is intentionally not used here. The notebook trains a fixed scikit-learn classifier on `train_pool` and evaluates it only on the time-based `holdout` loaded above.

### Tracking the fixed model with an experiment

In [ ]:
import mlflow

# Use the notebook default Fabric MLflow experiment. Explicit experiment
# lookup or creation requires additional workspace permissions.


#### Configure the multi-class classification model

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="lbfgs",
    random_state=41,
)


#### Fit the model and evaluate the time-based holdout

In [ ]:
from sklearn.metrics import f1_score, log_loss
from mlflow.models import infer_signature

with mlflow.start_run(run_name="ml_model_next_category_60d") as run:
    model.fit(X_train, y_train)
    holdout_predictions = model.predict(X_test)
    holdout_probabilities = model.predict_proba(X_test)
    holdout_macro_f1 = f1_score(y_test, holdout_predictions, average="macro")
    holdout_log_loss = log_loss(
        y_test,
        holdout_probabilities,
        labels=model.classes_,
    )

    mlflow.log_param("model_type", type(model).__name__)
    mlflow.log_param("feature_count", len(feature_cols))
    mlflow.log_metric("holdout_macro_f1", holdout_macro_f1)
    mlflow.log_metric("holdout_log_loss", holdout_log_loss)
    # Keep model logging safe when this cell is rerun after a kernel restart.
    from sklearn.utils.validation import check_is_fitted
    from sklearn.exceptions import NotFittedError

    try:
        check_is_fitted(model)
    except NotFittedError:
        model.fit(X_train, y_train)

    model_signature = infer_signature(X_train, model.predict(X_train.head(1)))
    mlflow.sklearn.log_model(
        model,
        "model",
        signature=model_signature,
        input_example=X_train.head(1),
    )

print(f"Holdout macro F1: {holdout_macro_f1:.4f}")
print(f"Holdout log loss: {holdout_log_loss:.4f}")


## Step 4: Save the trained machine learning model

Upon completing the standard model training, you can now save the trained model as an ML model in Fabric.

In [ ]:
model_path = f"runs:/{run.info.run_id}/model"

# Register the model to the MLflow registry
registered_model = mlflow.register_model(model_uri=model_path, name="ml_model_next_category_60d")

# Print the registered model's name and version
print(f"Model '{registered_model.name}' version {registered_model.version} registered successfully.")

## Step 5: Generate predictions

Microsoft Fabric lets you operationalize machine learning models with a scalable function called `PREDICT`, which supports batch scoring (or batch inferencing) in any compute engine. You can generate batch predictions directly from the Microsoft Fabric notebook or from a given ML model's item page. For more information on how to use `PREDICT`, see [Model scoring with PREDICT in Microsoft Fabric](https://aka.ms/fabric-predict).

1. Generate predictions.

In [ ]:
model_name = "ml_model_next_category_60d"
# SynapseML versions that use np.string_ are incompatible with NumPy 2.x.
# Keep the Fabric batch-scoring API working without downgrading the notebook runtime.
if not hasattr(np, "string_"):
    np.string_ = np.bytes_

from synapse.ml.predict import MLFlowTransformer

feature_cols = X_train.columns.to_list()
model = MLFlowTransformer(
    inputCols=feature_cols,
    outputCol=target_col,
    modelName=model_name,
    modelVersion=registered_model.version,
)

df_test = spark.createDataFrame(X_test)
batch_predictions = model.transform(df_test)


In [ ]:
display(batch_predictions)

2. Save the predictions to a table.

In [ ]:
saved_name = "Tables/dbo/ml_wide_next_category_60d_predictions".replace(".", "_")
batch_predictions.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(saved_name)